### Import libraries and load environment vairables (env.)

In [ ]:
import os
import json
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

### Beatify message helper function

In [ ]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [ ]:
bfmsgs(result)

### Create a FX function

In [ ]:
def get_fx(currency: str) -> float:
    """Return the KRW to currency exchange rate using the live USD-based API."""
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Missing EXCHANGE_RATE_API_KEY in .env")

    # Use USD as the base currency, then convert KRW to the requested currency.
    url = f"https://v6.exchangerate-api.com/v6/{api_key}/latest/USD"
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; hai5016-project/1.0)"},
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as error:
        raise RuntimeError(f"Failed to fetch exchange rates: {error}") from error

    if data.get("result") != "success":
        error_type = data.get("error-type", "unknown_error")
        raise RuntimeError(f"ExchangeRate API error: {error_type}")

    rates = data.get("conversion_rates", {})
    code = currency.upper()

    if "KRW" not in rates:
        raise RuntimeError("Exchange rate for KRW not found in API response")

    krw_rate = float(rates["KRW"])

    if code == "USD":
        return 1.0 / krw_rate

    if code not in rates:
        raise ValueError(f"Currency code not found: {code}")

    return float(rates[code]) / krw_rate

In [ ]:
# Test the get_fx function with a few currency codes.
# This checks the current behavior of the function defined above.

print("KRW:", get_fx("KRW"))
print("USD:", get_fx("USD"))
print("EUR:", get_fx("EUR"))

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
bfmsgs(result)

### Make a realworld tool, get website text

In [ ]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    text = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [ ]:
fetch_text_from_url("https://www.hanyang.ac.kr/re12")

In [ ]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information.
- `get_fx`: gets the current KRW exchange rate for a requested currency code."""

In [ ]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url, get_fx],
    system_prompt=SYSTEM_PROMPT
)

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at Hanyang University? Find it at https://www.hanyang.ac.kr/re12 and convert any KRW prices into USD."}]}
)
bfmsgs(result)